# Group-wise Comparisons and Differential Expression — Thymus Samples

Compare gene expression and cell type composition across three thymus Xenium samples:
- TMA172_THYMUS
- HDL065
- HDL073

**Input:** Three preprocessed AnnData objects (from notebook 01)

**Output:** Differential expression results and statistical comparisons across samples

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=80)
print(f"Scanpy version: {sc.__version__}")

Scanpy version: 1.11.5


In [ ]:
# Paths and parameters
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")
FIGURES_DIR = Path("../figures/04_group_comparisons")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = "THYMUS_comparison"

# Cluster key from notebook 01 — change to 'celltype' if annotation is added later
CLUSTER_KEY = "leiden_combined"

# Define the three thymus samples: (label, path to preprocessed h5ad)
SAMPLES = {
    "TMA172_THYMUS": DATA_DIR / "TMA172_THYMUS_Xenium.zarr" / "TMA172_THYMUS_xenium_preprocessed.h5ad",
    "HDL065": DATA_DIR / "HDL065_THYMUS_Xenium.zarr" / "HDL065_THYMUS_xenium_preprocessed.h5ad",
    "HDL073": DATA_DIR / "HDL073_THYMUS_Xenium.zarr" / "HDL073_THYMUS_xenium_preprocessed.h5ad",
}

# Load and concatenate
adatas = []
for sample_id, path in SAMPLES.items():
    a = sc.read_h5ad(path)
    a.obs['sample_id'] = sample_id
    adatas.append(a)
    print(f"Loaded {sample_id}: {a.shape}")

adata = ad.concat(adatas, label='sample_id', keys=list(SAMPLES.keys()), index_unique='-')
adata.obs['comparison_group'] = adata.obs['sample_id']

print(f"\nCombined dataset: {adata.shape}")
print(f"\nCells per sample:")
print(adata.obs['sample_id'].value_counts())

## 1. Data Integration and Joint Clustering

Re-normalize combined data from raw counts, select shared HVGs, integrate with Harmony to correct sample-level batch effects, and cluster jointly so cluster identities are shared across all samples.

## 1a. Subsample-Based Parameter Search

Subsample ~75k cells proportionally from each sample, then sweep Harmony + neighbor + UMAP
parameters to find a combination that produces a clean, well-separated UMAP without spikes.
Run the full-scale integration (Section 1b) only after confirming good parameters here.

In [ ]:
import gc
import time
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import connected_components

# ── Configuration ──────────────────────────────────────────────────
SUBSAMPLE_N = 65_000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Stratified subsample ──────────────────────────────────────────
sample_counts = adata.obs['sample_id'].value_counts()
sample_fracs  = sample_counts / sample_counts.sum()
sample_n      = (sample_fracs * SUBSAMPLE_N).round().astype(int)
sample_n.iloc[0] += SUBSAMPLE_N - sample_n.sum()  # adjust rounding

idx = []
for sid, n in sample_n.items():
    pool = adata.obs.index[adata.obs['sample_id'] == sid]
    idx.extend(np.random.choice(pool, size=min(n, len(pool)), replace=False))

adata_sub = adata[idx].copy()
print(f"Subsampled {adata_sub.n_obs:,} cells from {adata.n_obs:,}")
print(adata_sub.obs['sample_id'].value_counts())

# ── Preprocess subsample ──────────────────────────────────────────
adata_sub.X = adata_sub.layers['counts'].copy()
sc.pp.highly_variable_genes(
    adata_sub, n_top_genes=500, flavor='seurat_v3',
    batch_key='sample_id', subset=False
)
sc.pp.normalize_total(adata_sub, target_sum=None)
sc.pp.log1p(adata_sub)

adata_hvg_sub = adata_sub[:, adata_sub.var.highly_variable].copy()
# sc.pp.scale(adata_hvg_sub, max_value=10)
sc.pp.pca(adata_hvg_sub, n_comps=50, svd_solver='arpack')
adata_sub.obsm['X_pca'] = adata_hvg_sub.obsm['X_pca'].copy()
adata_sub.uns['pca'] = adata_hvg_sub.uns['pca'].copy()
del adata_hvg_sub; gc.collect()

# ── PCA scree plot ────────────────────────────────────────────────
pca_var = adata_sub.uns['pca']['variance_ratio']
cumvar  = np.cumsum(pca_var)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(pca_var)+1), pca_var, 'o-', markersize=3)
axes[0].set_xlabel('PC'); axes[0].set_ylabel('Variance ratio')
axes[0].set_title('PCA Scree Plot')
axes[0].axhline(0.01, ls='--', c='red', alpha=0.5)

axes[1].plot(range(1, len(cumvar)+1), cumvar, 'o-', markersize=3)
axes[1].set_xlabel('PC'); axes[1].set_ylabel('Cumulative variance')
axes[1].set_title('Cumulative Variance Explained')
for thresh in [0.80, 0.90, 0.95]:
    n_pc = int(np.searchsorted(cumvar, thresh)) + 1
    axes[1].axhline(thresh, ls='--', alpha=0.3)
    axes[1].annotate(f'{thresh:.0%} at PC {n_pc}', (n_pc, thresh), fontsize=8, ha='left')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'param_sweep_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Variance explained — 20 PCs: {cumvar[19]:.1%}, 30 PCs: {cumvar[29]:.1%}, 50 PCs: {cumvar[49]:.1%}")

# ── Phase 1: Harmony parameter sweep ─────────────────────────────
harmony_configs = [
    {'max_iter_harmony': 10, 'theta': 1.0},   # current defaults
    {'max_iter_harmony': 20, 'theta': 2.0},   # stronger correction
]

print("\n" + "=" * 60)
print("PHASE 1: Harmony parameter sweep (fixed nn=50, pcs=10, md=0.3)")
print("=" * 60)

harmony_results = {}
for hp in harmony_configs:
    label = f"theta={hp['theta']}, max_iter={hp['max_iter_harmony']}"
    print(f"\n  Testing: {label}")
    t0 = time.time()

    adata_test = adata_sub.copy()
    sc.external.pp.harmony_integrate(adata_test, key='sample_id', **hp)
    sc.pp.neighbors(adata_test, use_rep='X_pca_harmony', n_neighbors=30, n_pcs=10, metric='cosine')

    n_comp, _ = connected_components(adata_test.obsp['connectivities'], directed=False)
    print(f"    Connected components: {n_comp}")

    sc.tl.umap(adata_test, min_dist=0.3, spread=1.0, random_state=42)
    elapsed = time.time() - t0
    print(f"    Time: {elapsed:.1f}s")

    harmony_results[label] = adata_test

fig, axes = plt.subplots(len(harmony_results), 2, figsize=(14, 6 * len(harmony_results)))
if len(harmony_results) == 1:
    axes = axes[np.newaxis, :]
for i, (label, ad) in enumerate(harmony_results.items()):
    sc.pl.umap(ad, color='sample_id', ax=axes[i, 0], show=False, size=1)
    axes[i, 0].set_title(f'{label} — by sample')
    sc.pl.umap(ad, color='total_counts', ax=axes[i, 1], show=False, size=1)
    axes[i, 1].set_title(f'{label} — total counts')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'param_sweep_harmony.png', dpi=150, bbox_inches='tight')
plt.show()

del harmony_results; gc.collect()

# ── Phase 2: Neighbor + UMAP sweep ───────────────────────────────
# SET THESE based on Phase 1 results (defaults assume theta=2.0 wins)
BEST_HARMONY = {'max_iter_harmony': 20, 'theta': 2.0}

print("\n" + "=" * 60)
print("PHASE 2: Neighbor + UMAP parameter sweep")
print("=" * 60)

adata_harm = adata_sub.copy()
sc.external.pp.harmony_integrate(adata_harm, key='sample_id', **BEST_HARMONY)

neighbor_configs = [
    {'n_neighbors': 15, 'n_pcs': 10},
    {'n_neighbors': 30, 'n_pcs': 10},
    {'n_neighbors': 50, 'n_pcs': 10},
    {'n_neighbors': 15, 'n_pcs': 15},
    {'n_neighbors': 30, 'n_pcs': 15},
    {'n_neighbors': 50, 'n_pcs': 15},
]
umap_configs = [
    {'min_dist': 0.03, 'spread': 2.5},
    {'min_dist': 0.1, 'spread': 2.5},
]

results = {}
for np_cfg in neighbor_configs:
    for up_cfg in umap_configs:
        label = (f"nn={np_cfg['n_neighbors']}, pcs={np_cfg['n_pcs']}, "
                 f"md={up_cfg['min_dist']}")
        print(f"  {label}", end=" ... ")
        t0 = time.time()

        adata_test = adata_harm.copy()
        sc.pp.neighbors(adata_test, use_rep='X_pca_harmony', **np_cfg, metric='cosine')

        n_comp, _ = connected_components(adata_test.obsp['connectivities'], directed=False)
        sc.tl.umap(adata_test, random_state=42, **up_cfg)
        elapsed = time.time() - t0
        print(f"{elapsed:.1f}s, components={n_comp}")

        results[label] = adata_test

# Plot grid
n_combos = len(results)
ncols = 4
nrows = (n_combos + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes_flat = axes.flatten()
for i, (label, ad) in enumerate(results.items()):
    sc.pl.umap(ad, color='sample_id', ax=axes_flat[i], show=False,
               size=0.5, title=label, legend_loc='none')
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'param_sweep_neighbors_umap.png', dpi=150, bbox_inches='tight')
plt.show()

del results, adata_harm, adata_sub; gc.collect()
print("\nParameter sweep complete. Review plots and set BEST_PARAMS below.")

In [ ]:
# ── BEST PARAMETERS (update after reviewing sweep plots) ─────────
BEST_PARAMS = {
    # Harmony
    'max_iter_harmony': 20,
    'theta': 1.0,
    # Neighbors
    'n_neighbors': 20,
    'n_pcs': 15,       # elbow at ~10 PCs; beyond that is noise
    # UMAP
    'min_dist': 0.03,
    'spread': 2.5,
    # Leiden
    'resolution': 1.0,
}
print("Selected parameters for full-scale run:")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
import gc
from scipy.sparse.csgraph import connected_components

# --- 1b. Full-scale Joint Integration (using tuned parameters) ---
adata.X = adata.layers['counts'].copy()

# HVG selection per batch (uses raw counts with seurat_v3)
sc.pp.highly_variable_genes(
    adata, n_top_genes=2000, flavor='seurat_v3',
    batch_key='sample_id', subset=False
)
print(f'Highly variable genes: {adata.var.highly_variable.sum()}')

# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=None)
sc.pp.log1p(adata)
adata.layers['log_normalized'] = adata.X.copy()

# PCA on HVGs only
adata_hvg = adata[:, adata.var.highly_variable].copy()
# sc.pp.scale(adata_hvg, max_value=10)
sc.pp.pca(adata_hvg, n_comps=max(BEST_PARAMS['n_pcs'], 50), svd_solver='arpack')
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca'].copy()
adata.uns['pca'] = adata_hvg.uns['pca'].copy()
del adata_hvg
gc.collect()

# Harmony batch correction with tuned parameters
print('Running Harmony integration...')
sc.external.pp.harmony_integrate(
    adata, key='sample_id',
    max_iter_harmony=BEST_PARAMS['max_iter_harmony'],
    theta=BEST_PARAMS['theta'],
)
print('Harmony done.')

# Neighbors on corrected embedding
sc.pp.neighbors(
    adata, use_rep='X_pca_harmony',
    n_neighbors=BEST_PARAMS['n_neighbors'],
    n_pcs=BEST_PARAMS['n_pcs'],
    metric='cosine'
)

# Connectivity check
n_components, _ = connected_components(adata.obsp['connectivities'], directed=False)
print(f'Neighbor graph connected components: {n_components}')
if n_components > 1:
    print('WARNING: graph has disconnected components — UMAP may show spikes.')
    print('Consider increasing n_neighbors or n_pcs.')

# UMAP and clustering
sc.tl.umap(
    adata,
    min_dist=BEST_PARAMS['min_dist'],
    spread=BEST_PARAMS['spread'],
    random_state=42,
)
sc.tl.leiden(adata, resolution=BEST_PARAMS['resolution'], key_added=CLUSTER_KEY)

print(f'\nJoint clusters: {adata.obs[CLUSTER_KEY].nunique()}')
print(f'Cells per cluster:')
print(adata.obs[CLUSTER_KEY].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
sc.pl.umap(adata, color='sample_id', ax=axes[0], show=False, size=0.5)
axes[0].set_title('By Sample (after Harmony)')
sc.pl.umap(adata, color=CLUSTER_KEY, ax=axes[1], show=False, size=0.5,
           legend_loc='on data', legend_fontsize=6)
axes[1].set_title('Joint Leiden Clusters')
sc.pl.umap(adata, color='total_counts', ax=axes[2], show=False, size=0.5)
axes[2].set_title('Total Counts')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'umap_integrated.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Define Comparison Groups

Define groups for comparison (condition, treatment, region, etc.)

In [ ]:
# Comparison groups are the three thymus samples
print("Comparison groups:")
print(adata.obs['comparison_group'].value_counts())

print(f"\nCluster key: {CLUSTER_KEY}")
print(f"Clusters per sample:")
print(adata.obs.groupby('sample_id')[CLUSTER_KEY].nunique())

## 3. Cell Type Composition Analysis

In [ ]:
# Calculate cluster proportions per sample
composition = pd.crosstab(
    adata.obs['comparison_group'],
    adata.obs[CLUSTER_KEY],
    normalize='index'
) * 100

print(f"\n{CLUSTER_KEY} composition (%) by sample:")
print(composition)

# Plot composition
composition.T.plot(kind='bar', figsize=(14, 6))
plt.ylabel('Percentage')
plt.xlabel(CLUSTER_KEY)
plt.title(f'{CLUSTER_KEY} Composition by Sample')
plt.legend(title='Sample')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_composition.png', dpi=300, bbox_inches='tight')
plt.show()

# Save composition data
composition.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_cluster_composition.csv")

## 4. Statistical Testing of Cell Type Abundance

In [ ]:
# Chi-square test for cluster distribution across samples
from scipy.stats import chi2_contingency

ct_counts = pd.crosstab(adata.obs['comparison_group'], adata.obs[CLUSTER_KEY])
chi2, pval, dof, expected = chi2_contingency(ct_counts)

print(f"\nChi-square test for {CLUSTER_KEY} distribution across samples:")
print(f"  Chi-square statistic: {chi2:.4f}")
print(f"  P-value: {pval:.4e}")
print(f"  Degrees of freedom: {dof}")

if pval < 0.05:
    print("  Result: Significant difference in cluster distribution across samples")
else:
    print("  Result: No significant difference in cluster distribution across samples")

## 5. Differential Expression Analysis

In [ ]:
# Subsample proportionally for DE (preserves relative sample sizes)
np.random.seed(42)
frac = min(1.0, 10000 / adata.n_obs)  # target ~10k cells total
indices = []
for group in adata.obs['comparison_group'].unique():
    group_idx = adata.obs.index[adata.obs['comparison_group'] == group]
    n_sample = max(50, int(len(group_idx) * frac))  # at least 50 per group
    n_sample = min(n_sample, len(group_idx))
    indices.extend(np.random.choice(group_idx, size=n_sample, replace=False))
adata_de = adata[indices].copy()
print(f'Subsampled for DE: {adata_de.shape[0]} cells ({adata_de.obs["comparison_group"].value_counts().to_dict()})')

# Perform DE between samples (.X is log-normalized from integration)
sc.tl.rank_genes_groups(
    adata_de,
    groupby='comparison_group',
    method='wilcoxon',
    key_added='de_group',
    use_raw=False
)

# Plot top DE genes
sc.pl.rank_genes_groups(adata_de, n_genes=25, sharey=False, key='de_group')
plt.savefig(FIGURES_DIR / 'de_genes_by_sample.png', dpi=300, bbox_inches='tight')
plt.show()

# Extract DE results
de_results = sc.get.rank_genes_groups_df(adata_de, group=None, key='de_group')
de_results.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_de_genes_by_sample.csv", index=False)
print(f"\nDE results saved with {len(de_results)} entries")
print(f"logfoldchanges NaN: {de_results['logfoldchanges'].isna().sum()}")
print(f"pvals_adj == 0: {(de_results['pvals_adj'] == 0).sum()} / {len(de_results)}")

## 6. Cluster-Specific Differential Expression

DE between samples within each cluster

In [ ]:
# Perform DE within each cluster across samples
de_cluster_results = []

for cluster in sorted(adata.obs[CLUSTER_KEY].unique()):
    print(f"\nAnalyzing cluster {cluster}...")

    adata_ct = adata[adata.obs[CLUSTER_KEY] == cluster].copy()

    if adata_ct.obs['comparison_group'].nunique() < 2:
        print(f"  Skipping cluster {cluster}: not enough groups")
        continue

    if adata_ct.n_obs < 10:
        print(f"  Skipping cluster {cluster}: too few cells ({adata_ct.n_obs})")
        continue

    # Subsample proportionally within cluster
    np.random.seed(42)
    frac = min(1.0, 5000 / adata_ct.n_obs)  # target ~5k cells per cluster
    idx = []
    for group in adata_ct.obs['comparison_group'].unique():
        group_idx = adata_ct.obs.index[adata_ct.obs['comparison_group'] == group]
        n_sample = max(50, int(len(group_idx) * frac))
        n_sample = min(n_sample, len(group_idx))
        idx.extend(np.random.choice(group_idx, size=n_sample, replace=False))
    adata_sub = adata_ct[idx].copy()

    sc.tl.rank_genes_groups(
        adata_sub,
        groupby='comparison_group',
        method='wilcoxon',
        key_added='de',
        use_raw=False
    )

    ct_de = sc.get.rank_genes_groups_df(adata_sub, group=None, key='de')
    ct_de[CLUSTER_KEY] = cluster
    de_cluster_results.append(ct_de)

    print(f"  Found {len(ct_de)} DE genes ({adata_sub.n_obs} cells used)")

if de_cluster_results:
    all_de_ct = pd.concat(de_cluster_results, ignore_index=True)
    all_de_ct.to_csv(
        OUTPUT_DIR / f"{SAMPLE_NAME}_de_genes_by_cluster.csv",
        index=False
    )
    print(f"\nCluster-specific DE results saved: {len(all_de_ct)} total entries")

## 7. Volcano Plots

In [ ]:
from adjustText import adjust_text

# Create volcano plots — one per sample (each vs rest)
groups = de_results['group'].unique()

fig, axes = plt.subplots(1, len(groups), figsize=(7 * len(groups), 7))
if len(groups) == 1:
    axes = [axes]

pval_thresh = 0.05
fc_thresh = 1.0

for ax, group in zip(axes, groups):
    de_group = de_results[de_results['group'] == group].dropna(subset=['logfoldchanges']).copy()
    de_group['-log10_pval'] = -np.log10(de_group['pvals_adj'].clip(lower=1e-300))

    is_up = (de_group['pvals_adj'] < pval_thresh) & (de_group['logfoldchanges'] >= fc_thresh)
    is_down = (de_group['pvals_adj'] < pval_thresh) & (de_group['logfoldchanges'] <= -fc_thresh)
    is_nonsig = ~(is_up | is_down)

    ax.scatter(de_group.loc[is_nonsig, 'logfoldchanges'], de_group.loc[is_nonsig, '-log10_pval'],
               alpha=0.4, s=8, c='gray', label='NS', zorder=1)
    ax.scatter(de_group.loc[is_up, 'logfoldchanges'], de_group.loc[is_up, '-log10_pval'],
               alpha=0.7, s=12, c='firebrick', label=f'Up ({is_up.sum()})', zorder=2)
    ax.scatter(de_group.loc[is_down, 'logfoldchanges'], de_group.loc[is_down, '-log10_pval'],
               alpha=0.7, s=12, c='steelblue', label=f'Down ({is_down.sum()})', zorder=2)

    # Label top 10 by |LFC| + top 5 by p-value, deduplicated
    sig_all = de_group[is_up | is_down]
    top_lfc = sig_all.reindex(sig_all['logfoldchanges'].abs().nlargest(10).index)
    top_pval = sig_all.nlargest(5, '-log10_pval')
    to_label = pd.concat([top_lfc, top_pval]).drop_duplicates(subset='names')

    texts = []
    for _, gene in to_label.iterrows():
        texts.append(ax.text(gene['logfoldchanges'], gene['-log10_pval'],
                             gene['names'], fontsize=7, alpha=0.85))

    ax.axhline(-np.log10(pval_thresh), color='black', linestyle='--', alpha=0.4, lw=0.8)
    ax.axvline(-2.0, color='black', linestyle='--', alpha=0.4, lw=0.8)
    ax.axvline(2.0, color='black', linestyle='--', alpha=0.4, lw=0.8)
    ax.set_xlim(-8, 8)

    if len(sig_all) > 0:
        y_top = sig_all['-log10_pval'].max() * 1.15
    else:
        y_top = -np.log10(pval_thresh) + 2
    y_top = max(y_top, -np.log10(pval_thresh) + 2)
    ax.set_ylim(0, y_top)
    ax.set_xlabel('Log2 Fold Change')
    ax.set_ylabel('-Log10 Adjusted P-value')
    ax.set_title(f'{group} vs rest')
    ax.legend(fontsize=8, loc='upper right')

    if texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5, lw=0.5))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'volcano_plots.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Heatmap of Top DE Genes

In [ ]:
# Collect all significant DE genes shown in volcano plots
sig = de_results[
    (de_results['pvals_adj'] < 0.05) &
    (de_results['logfoldchanges'].abs() >= fc_thresh)
].copy()

# Rank by |LFC| within each group, take top 20 per group
top_per_group = (
    sig.assign(abs_lfc=sig['logfoldchanges'].abs())
    .sort_values('abs_lfc', ascending=False)
    .groupby('group')
    .head(20)
)
top_de_genes = top_per_group['names'].unique().tolist()
print(f'Heatmap genes: {len(top_de_genes)} (top 20 significant per group)')

# Compute mean expression per group, then z-score per gene
import scipy.sparse as sp
from scipy.stats import zscore

expr = adata[:, top_de_genes].X
if sp.issparse(expr):
    expr = expr.toarray()
expr_df = pd.DataFrame(expr, index=adata.obs_names, columns=top_de_genes)
expr_df['group'] = adata.obs['comparison_group'].values
mean_expr = expr_df.groupby('group').mean()
z = mean_expr.apply(zscore, axis=0).fillna(0)

# Diverging heatmap clipped at ±2 SD
fig, ax = plt.subplots(figsize=(max(8, len(mean_expr) * 2), max(10, len(top_de_genes) * 0.3)))
sns.heatmap(
    z.T,
    cmap='RdBu_r',
    center=0,
    vmin=-2, vmax=2,
    xticklabels=True,
    yticklabels=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('DE Genes: Mean Expression Z-scores by Sample')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'de_genes_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Gene Set Enrichment (if applicable)

In [ ]:
# Export gene lists per sample for external enrichment analysis
sig_genes = de_results[de_results['pvals_adj'] < 0.05]

for group in sig_genes['group'].unique():
    group_sig = sig_genes[sig_genes['group'] == group]
    
    upregulated = group_sig[group_sig['logfoldchanges'] > 1]['names'].tolist()
    downregulated = group_sig[group_sig['logfoldchanges'] < -1]['names'].tolist()
    
    with open(OUTPUT_DIR / f"{SAMPLE_NAME}_{group}_upregulated_genes.txt", 'w') as f:
        f.write('\n'.join(upregulated))
    
    with open(OUTPUT_DIR / f"{SAMPLE_NAME}_{group}_downregulated_genes.txt", 'w') as f:
        f.write('\n'.join(downregulated))
    
    print(f"{group}: {len(upregulated)} upregulated, {len(downregulated)} downregulated")

print(f"\nGene lists saved for external enrichment analysis")

## 10. Summary Report

In [ ]:
# Generate summary
sig_all = de_results[de_results['pvals_adj'] < 0.05]

summary = {
    'Total cells': adata.n_obs,
    'Samples compared': adata.obs['comparison_group'].nunique(),
    'Clusters': adata.obs[CLUSTER_KEY].nunique(),
    'Total DE genes (p<0.05)': len(sig_all),
}

# Per-sample counts
for group in sorted(adata.obs['comparison_group'].unique()):
    n_cells = (adata.obs['comparison_group'] == group).sum()
    group_sig = sig_all[sig_all['group'] == group] if 'group' in sig_all.columns else pd.DataFrame()
    n_up = len(group_sig[group_sig['logfoldchanges'] > 1]) if len(group_sig) > 0 else 0
    n_down = len(group_sig[group_sig['logfoldchanges'] < -1]) if len(group_sig) > 0 else 0
    summary[f'{group} cells'] = n_cells
    summary[f'{group} upregulated'] = n_up
    summary[f'{group} downregulated'] = n_down

summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['Value']

print("\n=== Thymus Group Comparison Summary ===")
print(summary_df)

summary_df.to_csv(OUTPUT_DIR / f"{SAMPLE_NAME}_summary.csv")